# 03 · Classical course baseline: MFCC + GMM-HMM

Runs on Kaggle **on CPU** (no GPU needed). **Settings → Internet: On.** No manual uploads.

`scripts/train_gmm_hmm.py` trains on the train split (flat start, Viterbi re-estimation, mixture
splitting), chooses the LM weight and insertion penalty on dev, and decodes test once at the end
(settings: `configs/train_gmm_hmm.yaml`), in a virtual environment with the pinned packages of
`requirements/train.txt`. `TRIAL = True` uses 300 train clips and 20 dev/test clips.

Output: `/kaggle/working/results_gmm_hmm.zip` (predictions, `.meta.json`, training log).

In [ ]:
REPO_URL = "https://github.com/eneserman/NLP.git"  # the public repository
BRANCH = "main"
TRIAL = False
WORKERS = 4

In [ ]:
# The repository and a virtual environment with only our pinned packages live in /tmp, so that the
# preinstalled packages of the Kaggle image cannot conflict and /kaggle/working holds only the results.
import os
os.environ.pop("PYTHONPATH", None)          # nothing from the Kaggle image on the path of our environment
%cd /tmp
!rm -rf repo venv && git clone -q -b $BRANCH $REPO_URL repo
%cd /tmp/repo
!git log -1 --format='commit %H'
!pip install -q virtualenv==21.15.9 && python -m virtualenv -q /tmp/venv
PY = "/tmp/venv/bin/python"
!{PY} -m pip install -q -r requirements/train.txt
!{PY} -m pip freeze > /kaggle/working/pip_freeze.txt && cat /kaggle/working/pip_freeze.txt
!{PY} -c "import sys; print('sys.path:', sys.path)"
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "no GPU"

In [ ]:
!{PY} -m pytest -q tests/test_gmm_hmm.py

In [ ]:
!{PY} scripts/unpack_audio.py

In [ ]:
import subprocess
extra = ["--limit-train", "300", "--limit-eval", "20", "--out-dir", "/kaggle/working/trial"] if TRIAL else []
r = subprocess.run([PY, "scripts/train_gmm_hmm.py", "--workers", str(WORKERS), *extra])
print("exit code", r.returncode)

In [ ]:
if TRIAL:
    !head -c 3000 /kaggle/working/trial/train_logs/gmm_hmm.json
else:
    !cp work/models/gmm_hmm/model.npz /kaggle/working/gmm_hmm_model.npz
    !zip -q -r /kaggle/working/results_gmm_hmm.zip results/predictions/gmm_hmm.* results/train_logs/gmm_hmm.json
    !ls -lh /kaggle/working